# Control policy optimization Mujoco

## Set-up

### Imports

In [1]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Fri Jun 12 13:52:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.51                 Driver Version: 555.97         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   61C    P8              1W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title Import packages for plotting and creating graphics
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [8]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
import jax.numpy as jnp
import jax.random as jr
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

print("These device(s) are detected:", jax.devices())

These device(s) are detected: [CudaDevice(id=0)]


### Functions

In [3]:
from mujoco_playground import registry

class MujocoGymnaxWrapper:

    def __init__(self, env_name = None, env_instance = None, config_overrides = None):
        if env_instance is not None:
            self.env = env_instance
        elif config_overrides is not None:
            self.env = registry.load(env_name, config_overrides=config_overrides)
        else:
            self.env = registry.load(env_name)
        # self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, key, state, action, params=None):

      action = action.reshape(self.action_space)
      next_state = self.env.step(state, action)

      done = next_state.done
      obs = next_state.obs
      reward = next_state.reward

      return obs, next_state, reward, done, {}

    def render(self, rollout):
      return self.env.render(rollout)

    @property
    def dt(self):
        return self.env.dt

In [4]:
def compute_trajectory_py(key, env, policy, T=1000, stride=1):
  jit_reset = jax.jit(env.reset)
  jit_step = jax.jit(env.step)
  obs, env_state = jit_reset(key)
  states = []

  for t in range(T):
      action = policy(obs)
      obs, env_state, reward, done, _ = jit_step(key, env_state, action)

      if t % stride == 0:
          states.append(env_state)

      if done:
          break

  return states

## Cartpole

In [6]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 15

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu')

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 15

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu')

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 16.18 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 12.60 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 4.54 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 4.75 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 4.40 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 3.74 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 2.72 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 115.45 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load 

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 51.90 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -366.75347900390625, equation: [-0.0638]
Complexity: 3, fitness: -547.6666259765625, equation: [-0.0775*y3]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equation: [0]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equa

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Reacher

In [9]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy', config_overrides={'njmax': 500, 'naconmax': 500})

### 25 generations

In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [10]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

Module update_gradient_JTDAJ_dense_tiled__locals__kernel_1131f197 70eab00 load on device 'cuda:0' took 23.22 ms  (cached)
Module linesearch_iterative__locals__kernel_d0fc91ea fbe7a03 load on device 'cuda:0' took 6.04 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 18a78e5 load on device 'cuda:0' took 9.69 ms  (cached)
Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5'].


In [11]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 45.10 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -49.5, equation: [-0.0638, 0]
Complexity: 2, fitness: -62.5, equation: [sin(y4), 0]
Complexity: 3, fitness: -98.0, equation: [y0*y3, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fitness: -275.5, equation: [-y3, y2 + y3]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
Complexity: 9, fitness: -410.75, equation: [y1*y3 + y2, 2*y1*y3 + y2 - 0.0753*y5]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -149.0625, equation: [y3, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fitness: -275.5, equation: [-y3, y2 + y3]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
Complexity: 9, fitness: -410.75, equation: [y1*y3 + y2, 2*y1*y3 + y2 - 0

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Swimmer

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

In [ ]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

### 25 generations

In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [5]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

E0612 13:53:05.673337    1839 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0612 13:53:05.689389    1559 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [6]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 4.95 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 11.53 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 5.47 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 18a78e5 load on device 'cuda:0' took 5.02 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 3.96 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 4.32 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 4.03 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 3.46 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f9126636 4908da2 load on device 'cuda:0' took 109.92 ms  (cached)
Module m

In [10]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 88.45 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -159.62271118164062, equation: [0, 0.639]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 4, fitness: -167.80514526367188, equation: [sin(y2), y8 - sin(y2)]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
Complexity: 11, fitness: -176.5635986328125, equation: [y12 + y2*y9*(y12 - 0.684*y2) - 0.684*y2 + y3, y2*y9*(y12 - 0.684*y2) - 0.684*y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 3, fitness: -167.28628540039062, equation: [y3 + y9, 0]
Complexity: 4, fitness: -170.99008178710938, equation: [y11 + 2

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Hopper

In [11]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')

### 25 generations

In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4)

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [12]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4)

Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 15.24 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 5.25 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_052aeb65 052aeb6 load on device 'cuda:0' took 4.40 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 105.22 ms  (cached)
Module solve_init_jaref__locals__kernel_173cbb76 173cbb7 load on device 'cuda:0' took 4.57 ms  (cached)
Module mul_m_dense__locals___mul_m_dense_d21c2e6d d21c2e6 load on device 'cuda:0' took 2.96 ms  (cached)
Module update_constraint_gauss_cost__locals__kernel_b1449efe b1449ef load on device 'cuda:0' took 3.42 ms  (cached)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_83d2c797 52c34bc load on device 'cuda:0' took 4.76 ms  (cached)
Module update_gradient_cholesky__locals__kernel_6f68b037 18ece50 loa

In [13]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 89.11 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.004872772842645645, equation: [0, 0.639, 0, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.3909245431423187, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -1.173608660697937, equation: [0, y10 + y5 + y8, 0, 0]
Complexity: 9, fitness: -1.9155478477478027, equation: [0, y1 + y10 + 2*y2**2*(y1 + y10), 2*y2, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -0.059083644300699234, equation: [0, 0, y7, 0]
Complexity: 3, fitness: -0.4087880849838257, equation: [0, y10 + 0.352, 0, 0]
Complexity: 5, fitness: -1.173608660697937, equation: [0, y10 + y5 + y8, 0, 0]
Complexity: 9, fitness: -1.9155478477478027, equation: [0, y1 + y10 + 2*y2**2*(y1 + y10), 2*y2, 0]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -0.523652672767

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Walker

In [14]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')

In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [15]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 117.27 ms  (cached)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 4.54 ms  (cached)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 load on device 'cuda:0' took 2.68 ms  (cached)
Module update_constraint_gauss_cost__locals__kernel_0aec6e48 0aec6e4 load on device 'cuda:0' took 2.93 ms  (cached)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_2b23959a 84da6db load on device 'cuda:0' took 4.63 ms  (cached)
Module update_gradient_cholesky__locals__kernel_97d1b30b 6b32db2 load on device 'cuda:0' took 262.36 ms  (cached)
Module linesearch_iterative__locals__kernel_6962ca73 89f17c9 load on device 'cuda:0' took 5.45 ms  (cached)
Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23'].

In [16]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 95.14 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -24.646881103515625, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 3, fitness: -44.86970520019531, equation: [0, 0, 0, 0.24*y21, 0, 0]
Complexity: 5, fitness: -58.453704833984375, equation: [y3*(y15 + y22), 0, 0, 0, y15 + y22, 0]
Complexity: 7, fitness: -68.48965454101562, equation: [0, y0*y5*y9, 0, y0*y5*y9 + y17, 0, y0*y9]
Complexity: 9, fitness: -68.63204956054688, equation: [0, y10*y22*(-y13 + y2), 0, y10*y22*(-y13 + y2) + y10*y22 + y18, 0, 0]
Complexity: 11, fitness: -95.50133514404297, equation: [y16 + y6 + (y15 + y23)*(-y3 + y6), y16 + y6, y15 + y23, (y15 + y23)*(-y3 + y6), -y3 + y6, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -34.13189697265625, equation: [0, 0, 0, y18, 0, 0]
Complexity: 3, fitness: -58.457847595214844, equation: [0, 0, 0, y12 + y17, 0, 0]
Complexity: 5, fitness: -7

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Cheetah

In [17]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')

In [ ]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [18]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16'].


In [19]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.5
Compiling code for evaluation and evolution...
Finished compilation in 47.20 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.5508462190628052, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 2, fitness: -15.399696350097656, equation: [0, 0, 0, 0, sin(y13), 0]
Complexity: 3, fitness: -75.71392822265625, equation: [y6 + y8, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -85.68940734863281, equation: [y11 + y16 + y9, 0, 0, 0, 0, y11 + y16 + 2*y9]
Complexity: 8, fitness: -112.31211853027344, equation: [-2*y16 - 3*y6 + 2*cos(y7), 0, 0, 0, cos(y7), y16 + y6]
Complexity: 9, fitness: -144.18373107910156, equation: [y0 + y11 + y14*y7 + y16, 0, 0, 0, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -59.273441314697266, equation: [y8, 0, 0, 0, 0, 0]
Complexity: 3, fitness: -86.6945571899414, equation: [y11 + y6, 0, 0, 0, 0, 0]
Complexity: 5, fitness: -97.0964126586914, equation: [y11 + y8, 0, 0, 0, y13*(y11 + y8), 0]
Complexity: 6, 

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Multiple evaluations

In [ ]:
T = 1000

def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)
        (_, _, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn(policy, env),
            (key, obs, env_state, False),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

def step_fn(policy, env):

    def _step(carry, t):
        key, obs, env_state, done = carry
        action = policy(obs)
        key, subkey = jr.split(key)
        obs, env_state, reward, done_new, _ = env.step(
            subkey,
            env_state,
            action,
            None
        )

        return (key, obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

### CartPole

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Reacher

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Swimmer

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Hopper

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Walker

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Cheetah

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)